# Tutorial: Spatial site selection

This tutorial demonstrates four fast ways to select STREAM monitoring sites: bounding box, search radius, nearest sites, and Hydrologic Unit Code (HUC). These methods filter the small site metadata and HUC lookup tables; they do not download the large time-series datasets.

## 1. Connect to HydroShare

In [ ]:
from streamwq.client import StreamClient
from streamwq.gage_metadata_map import plot_gage_metadata_map

stream = StreamClient()

## 2. Select sites within a bounding box

A bounding box is written as `(west, south, east, north)` using longitude and latitude in WGS 84. This example covers part of the Mid-Atlantic region and also requires water-temperature observations.

In [ ]:
bbox_sites = stream.find_sites(
    variables="WTemp_C",
    bbox=(-80.0, 36.0, -75.0, 40.0),
)

print(f"Found {len(bbox_sites):,} sites inside the bounding box")
bbox_sites.head()

## 3. Select sites within a radius

The center is ordered as `(longitude, latitude)`. Results contain `distance_km` and are sorted from nearest to farthest.

In [ ]:
richmond = (-77.4360, 37.5407)

radius_sites = stream.find_sites(
    variables="WTemp_C",
    center=richmond,
    radius_km=100,
)

print(f"Found {len(radius_sites):,} sites within 100 km")
radius_sites[["STREAM_ID", "site_name", "distance_km"]].head(10)

## 4. Find the nearest sites

Use `nearest` when you need a fixed number of sites rather than every site inside a fixed distance. Existing filters are applied before distance ranking.

In [ ]:
nearest_sites = stream.find_sites(
    variables=["WTemp_C", "DO_mgL"],
    center=richmond,
    nearest=5,
)

nearest_sites[["STREAM_ID", "site_name", "distance_km"]]

You can combine `radius_km` and `nearest` to request at most five matching sites within 200 km.

In [ ]:
nearby_sites = stream.find_sites(
    center=richmond,
    radius_km=200,
    nearest=5,
)

nearby_sites[["STREAM_ID", "site_name", "distance_km"]]

## 5. Select sites by HUC6

HUC codes should be strings so leading zeros are preserved. HUC membership comes from the small `gage_huc_lookup.parquet` file in the STREAM HydroShare resource, so this operation does not query NHDPlus.

In [ ]:
huc6_sites = stream.find_sites(
    huc6="170900",
    variables="WTemp_C",
)

print(f"Found {len(huc6_sites):,} matching sites in HUC6 170900")
huc6_sites[["STREAM_ID", "site_name", "HUC6", "HUC8"]].head()

## 6. Refine the selection to a HUC8

Select the HUC8 containing the most matching sites in the HUC6 result, then run a HUC8 query. This avoids hardcoding a HUC8 that may not occur in a future dataset version.

In [ ]:
if huc6_sites.empty:
    raise ValueError("No matching sites were found in the example HUC6")

example_huc8 = huc6_sites["HUC8"].value_counts().index[0]
huc8_sites = stream.find_sites(
    huc8=example_huc8,
    variables="WTemp_C",
)

print(f"Found {len(huc8_sites):,} matching sites in HUC8 {example_huc8}")
huc8_sites[["STREAM_ID", "site_name", "HUC6", "HUC8"]]

## 7. Display the selected stations

The lightweight map below displays only the selected station points. Click a point to inspect its metadata.

In [ ]:
plot_gage_metadata_map(huc8_sites)

## 8. Use the spatial selection to retrieve observations

A spatial result is a regular DataFrame. Pass its `STREAM_ID` values directly to the time-series methods. Keep the first three sites and a short period for this example.

In [ ]:
selected_ids = huc8_sites["STREAM_ID"].head(3).tolist()

water_quality = stream.read_water_quality(
    site_ids=selected_ids,
    variables="WTemp_C",
    start="2020-01-01",
    end="2020-01-31 23:59:59",
    quality="approved",
)

print(f"Retrieved {len(water_quality):,} observations")
water_quality.head()

## Next steps

These selectors can be combined to build a study population before reading time-series data. A later tutorial can add clickable HUC boundaries after simplified boundary geometries are made available in HydroShare.